In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from dataclasses import dataclass, InitVar, field, asdict

### DATASET and PARAMETER

@dataclass(frozen=True)
class HyperParameter:
    batch_size: int
    num_epochs: int
    learning_rate: float

@dataclass
class MNIST:
    root: InitVar[str]
    param: InitVar[HyperParameter]

    train: DataLoader = field(init=False)
    test:  DataLoader = field(init=False)
    total: DataLoader = field(init=False)

    def __post_init__(self, root, param):
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,)),
        ])
        train_set = datasets.MNIST(
            root=root,
            train=True,
            download=True,
            transform=transform,
        )
        test_set = datasets.MNIST(
            root=root,
            train=False,
            download=True,
            transform=transform,
        )
        self.train = DataLoader(
            dataset=train_set,
            batch_size=param.batch_size, shuffle=True
        )
        self.test = DataLoader(
            dataset=test_set,
            batch_size=param.batch_size, shuffle=False
        )
        self.total = DataLoader(
            dataset=train_set+test_set,
            batch_size=param.batch_size, shuffle=False
        )

In [ ]:
### FIXED-POINT QAT: ap_fixed<16, 7> (W=16, I=7, F=9)
FIXED_W = 16
FIXED_I = 7
FIXED_F = FIXED_W - FIXED_I

FIXED_SCALE = 2 ** FIXED_F
FIXED_MIN = -(2 ** (FIXED_I - 1))
FIXED_MAX = (2 ** (FIXED_I - 1)) - (1 / FIXED_SCALE)


def fixed_point_round(x: torch.Tensor) -> torch.Tensor:
    x = torch.clamp(x, FIXED_MIN, FIXED_MAX)
    return torch.round(x * FIXED_SCALE) / FIXED_SCALE


class FixedPointQuantize(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        return fixed_point_round(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output


class QuantizedConv2d(nn.Conv2d):
    def forward(self, x):
        qw = FixedPointQuantize.apply(self.weight)
        qb = FixedPointQuantize.apply(self.bias) if self.bias is not None else None
        return self._conv_forward(x, qw, qb)


class QuantizedLinear(nn.Linear):
    def forward(self, x):
        qw = FixedPointQuantize.apply(self.weight)
        qb = FixedPointQuantize.apply(self.bias) if self.bias is not None else None
        return F.linear(x, qw, qb)


### NEURAL NETWORK DEFINITION (구조 동일, 레이어만 양자화 래핑)
class MNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = QuantizedConv2d(1, 16, 3, 1, bias=True)
        self.conv2 = QuantizedConv2d(16, 32, 3, 1, bias=True)
        self.conv3 = QuantizedConv2d(32, 32, 3, 1, bias=True)
        self.pool = nn.MaxPool2d(2)
        self.fc = QuantizedLinear(3872, 10, bias=True)

    def forward(self, x):
        x = FixedPointQuantize.apply(x)

        x = self.conv1(x)
        x = F.relu(x)
        x = FixedPointQuantize.apply(x)

        x = self.conv2(x)
        x = F.relu(x)
        x = FixedPointQuantize.apply(x)

        x = self.conv3(x)
        x = F.relu(x)
        x = FixedPointQuantize.apply(x)

        x = self.pool(x)
        x = FixedPointQuantize.apply(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x


In [10]:
import os
from pathlib import Path
from datetime import datetime

### TRAIN / EVALUATION ROUTINE

@dataclass
class Routine:
    device: object
    criterion: object
    optimizer: object

    def _routine(self, model, loader):
        num_hit, running_loss = 0, 0.0
        
        for images, labels in loader:
            images = images.to(self.device)
            labels = labels.to(self.device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            loss = self.criterion(outputs, labels)

            if torch.is_grad_enabled():
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()

            num_hit += (predicted==labels).sum().item()
            running_loss += loss.item()
            
        return num_hit / len(loader.dataset), \
               running_loss / len(loader)

    def train(self, model, loader):
        model.train()
        return self._routine(model, loader)

    def test(self, model, loader):
        model.eval()
        with torch.no_grad():
            return self._routine(model, loader)
            
@dataclass
class Milestone:
    parent: str | Path
    model_name: str
    ext: str
    now: str = field(init=False)

    def __post_init__(self):
        self.now = datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
        self.parent = Path(self.parent)

        if not self.parent.exists():
            self.parent.mkdir()

    def __iter__(self):
        files = self.parent.glob(f"{self.model_name}-*.{self.ext}")
        return iter(sorted(files, key=lambda f: f.name, reverse=True))

    def save(self, **kwargs):
        torch.save(kwargs, self.parent / f"{self.model_name}-{self.now}.{self.ext}")

In [11]:
### TRAIN / EVALUATION ROUTINE
# CUDA(리눅스/윈도 GPU) → MPS(애플 실리콘) → CPU 순으로 선택
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
param = HyperParameter(
    batch_size=256,
    num_epochs=10,
    learning_rate=1e-3
)
mnist = MNIST('./data', param)
model = MNISTCNN().to(device)

criterion =  nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=param.learning_rate)
routine = Routine(device, criterion, optimizer)

In [12]:
milestone = Milestone("./milestones_qat", type(model).__name__, 'pth')
print(f"Train Starts at: {milestone.now}")
print(f"Target Model: {milestone.model_name}")
print('')
print(f"| {'EPOCH': ^5} | {'TRAIN': ^33} | {'TEST': ^33} |")
print(f"| {'     ': ^5} "
      f"| {'ACCURACY': ^15} | {'LOSS': ^15} "
      f"| {'ACCURACY': ^15} | {'LOSS': ^15} |")

checkpoint = None
for epoch in range(param.num_epochs):
    train_acc, train_loss = routine.train(model, mnist.train)
    test_acc, test_loss = routine.test(model, mnist.test)

    print(f"| {epoch: ^5} "
          f"| {train_acc: ^15.2%} | {train_loss: ^15.5} "
          f"| {test_acc: ^15.2%} | {test_loss: ^15.5} |", end='')

    if (checkpoint is None) or (checkpoint['accuracy'] < test_acc):
        checkpoint = dict(
            epoch=epoch,
            accuracy=test_acc,
            param=asdict(param),
            model=model.state_dict(),
            optim=optimizer.state_dict(),
        )
        milestone.save(**checkpoint)
        print(' *', end='')
    print('')

Train Starts at: 2026-05-17-21-58-17
Target Model: MNISTCNN

| EPOCH |               TRAIN               |               TEST                |
|       |    ACCURACY     |      LOSS       |    ACCURACY     |      LOSS       |
|   0   |     91.77%      |     0.27961     |     98.11%      |    0.062345     | *
|   1   |     98.02%      |    0.064022     |     98.30%      |    0.051737     | *
|   2   |     98.58%      |    0.047504     |     98.50%      |    0.044247     | *
|   3   |     98.89%      |    0.036817     |     98.59%      |     0.03828     | *
|   4   |     99.06%      |    0.029981     |     98.56%      |    0.042009     |
|   5   |     99.17%      |    0.025409     |     98.81%      |    0.035982     | *
|   6   |     99.30%      |    0.021545     |     98.74%      |    0.041784     |
|   7   |     99.41%      |    0.018125     |     98.57%      |     0.04125     |
|   8   |     99.50%      |    0.015583     |     98.79%      |     0.03813     |
|   9   |     99.56%      |

In [13]:
model_name = 'MNISTCNN'
test_ms = Milestone(milestone.parent, model_name, milestone.ext)
test_model = globals()[model_name]()
test_model.eval()

with torch.no_grad():
    for ms_file in test_ms:
        checkpoint = torch.load(ms_file)
    
        test_model.load_state_dict(checkpoint['model'])
        test_model.to(device)
        accuracy, _ = routine.test(test_model, mnist.total)
        print(ms_file.stem, ':\t', f"{accuracy: .2%}")

MNISTCNN-2026-05-17-21-58-17 :	  99.63%


In [14]:
### 가중치 내보내기 (16/7 QAT → .npy)
# 위 셀들에서 `fixed_point_round`, `FIXED_*` 정의된 뒤 실행할 것.
import numpy as np
from pathlib import Path

# 내보낼 체크포인트 (None이면 milestones_qat 안 최신 MNISTCNN-*.pth)
EXPORT_CKPT: Path | None = None

ORDER = [
    "conv1.weight", "conv1.bias",
    "conv2.weight", "conv2.bias",
    "conv3.weight", "conv3.bias",
    "fc.weight", "fc.bias",
]


def _tensor_int16_ngrid(t: torch.Tensor) -> np.ndarray:
    q = fixed_point_round(t.detach().cpu())
    qi = torch.round(q * FIXED_SCALE).to(torch.int32)
    qi = torch.clamp(qi, -32768, 32767).to(torch.int16)
    return qi.numpy()


def _pick_ckpt() -> Path:
    if EXPORT_CKPT is not None:
        return Path(EXPORT_CKPT)
    files = sorted(Path("./milestones_qat").glob("MNISTCNN-*.pth"))
    if not files:
        raise FileNotFoundError("milestones_qat/*.pth 없음")
    return files[-1]


ckpt_path = _pick_ckpt()
out_dir = Path("./artifacts/cnn_qat_npy") / ckpt_path.stem
out_dir.mkdir(parents=True, exist_ok=True)

ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
sd = ckpt["model"]
lines = [f"checkpoint: {ckpt_path.resolve()}",
         f"ap_fixed<{FIXED_W},{FIXED_I}> fraction_bits(F)={FIXED_F}, scale=int 값 / {FIXED_SCALE}",
         "Conv2d weight: PyTorch [out_ch, in_ch, kH, kW]",
         "Linear weight: PyTorch [out_features, in_features]",
         ""]

for name in ORDER:
    if name not in sd:
        raise KeyError(name)
    t = sd[name]
    safe = name.replace(".", "_")
    qf = fixed_point_round(t).detach().cpu().numpy().astype(np.float32)
    np.save(out_dir / f"{safe}_float32.npy", qf)
    np.save(out_dir / f"{safe}_int16.npy", _tensor_int16_ngrid(t))
    lines.append(f"{name}: shape={tuple(t.shape)} → {safe}_float32.npy / {safe}_int16.npy")

readme = "\n".join(lines)
(out_dir / "README_export.txt").write_text(readme + "\n", encoding="utf-8")
print(readme)
print(f"\n저장 위치: {out_dir.resolve()}")

checkpoint: /Users/ongda/Desktop/5-1/Tensor_Processor_Design_for_Image_Recognition/Final/milestones_qat/MNISTCNN-2026-05-17-21-58-17.pth
ap_fixed<16,7> fraction_bits(F)=9, scale=int 값 / 512
Conv2d weight: PyTorch [out_ch, in_ch, kH, kW]
Linear weight: PyTorch [out_features, in_features]

conv1.weight: shape=(16, 1, 3, 3) → conv1_weight_float32.npy / conv1_weight_int16.npy
conv1.bias: shape=(16,) → conv1_bias_float32.npy / conv1_bias_int16.npy
conv2.weight: shape=(32, 16, 3, 3) → conv2_weight_float32.npy / conv2_weight_int16.npy
conv2.bias: shape=(32,) → conv2_bias_float32.npy / conv2_bias_int16.npy
conv3.weight: shape=(32, 32, 3, 3) → conv3_weight_float32.npy / conv3_weight_int16.npy
conv3.bias: shape=(32,) → conv3_bias_float32.npy / conv3_bias_int16.npy
fc.weight: shape=(10, 3872) → fc_weight_float32.npy / fc_weight_int16.npy
fc.bias: shape=(10,) → fc_bias_float32.npy / fc_bias_int16.npy

저장 위치: /Users/ongda/Desktop/5-1/Tensor_Processor_Design_for_Image_Recognition/Final/artifacts/cnn